In [1]:
from pathlib import Path
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

ROOT = Path().resolve().parent
sys.path.append(str(ROOT))

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
def get_spark():
    return SparkSession.builder \
        .appName('TCC - Testes') \
        .getOrCreate()

spark = get_spark()

In [4]:
base_path = "../data/criminalidade/curated/veiculos_setor_parquet"

df = (
    spark.read
    .option("basePath", base_path)
    .option("mergeSchema", "true")
    .parquet(f"{base_path}/ano_trimestre=*")
)

df_ocorrencias = df.withColumn(
    "ano",
    F.regexp_extract(
        F.col("ano_trimestre"),
        r"(\d{4})",
        1
    ).cast("int")
)

df_ocorrencias = (
    df_ocorrencias
    .filter(
        F.col("ano").cast("int").between(2020, 2025)
    )
)

df_ocorrencias = df_ocorrencias.filter(df_ocorrencias.SJ_CD_SETOR.isNotNull())
# df_ocorrencias.printSchema()
df_ocorrencias.show(5, False)

+------------+--------------------+--------------------+--------------+------+-------+------+----------------------+-------------------+------------------------+-------------------+------------------+---------------+----------------------+-------------------+----------------------+------------+--------------+-----------+----------------+-------------+--------------------+------------------+-----------------+-------+-------------------+---------------------------------+-----------------+-----------------+-----------------+------------+------------------------+------------------+--------------------+--------------+----------+-------------+-----------------+---------------+---------------+-------------------+-----------+-----------------+-----------------+----------------------------+----------------------+-------------------+-------------+--------+--------+-----------+---------------+-----------+---------+----------+-----------------+------------+------------+--------+---------+---------

In [5]:
df_censo_final = spark.read.parquet('../data/censo/final/processed/setor')
df_censo = df_censo_final.filter(df_censo_final.nm_mun == 'São Paulo')
# df_censo.printSchema()
df_censo.show(5, False)

+---------------+---------+---------+-----------+-----------+-----------+-------+--------------------+-------+---------+---------+----------+-------------+----------------------------------+--------------------------------+----------------------+--------------------+----------------------+-------------------+--------------------------------------------+------------------+------------------+---------------------+-------------------+-----------+----------+--------------------+-------------------+-------------------+----------------------------+---------------------------------------+-----------------------------------------+-------------------------------------------+------------------------------------+-----------------+-----------------+--------------------------+----------------------------+-------------------------+---------------------------+----------------------------+------------------------+----------------------+-----------------------------+
|cd_setor       |cd_dist  |nm_dist 

In [19]:
from pyspark.sql import functions as F

# ============================================================
# 1. Filtrar apenas roubos e furtos de veículos com setor válido
# ============================================================

df_crimes_filtrado = (
    df_ocorrencias
    .filter(
        (F.col("is_roubo") == "true") |
        (F.col("is_furto") == "true")
    )
    .filter(F.col("SJ_CD_SETOR").isNotNull())
)

# ============================================================
# 2. Agregar crimes por setor
#    Para Moran's I, precisamos de 1 linha por setor censitário
# ============================================================

df_crimes_setor = (
    df_crimes_filtrado
    .groupBy(
        F.col("SJ_CD_SETOR").alias("cd_setor")
    )
    .agg(
        F.count("*").alias("qt_crimes"),
        F.sum(
            F.when(F.col("is_roubo") == "true", 1).otherwise(0)
        ).alias("qt_roubos"),
        F.sum(
            F.when(F.col("is_furto") == "true", 1).otherwise(0)
        ).alias("qt_furtos")
    )
)

# ============================================================
# 3. Criar base Moran a partir do Censo
#    Isso preserva setores sem ocorrência criminal
# ============================================================

df_moran = (
    df_censo
    .filter(F.col("nm_mun") == "São Paulo")
    .filter(F.col("qt_pessoas") >= 100)
    .join(
        df_crimes_setor,
        on="cd_setor",
        how="left"
    )
)

# ============================================================
# 4. Preencher setores sem crime com zero
# ============================================================

df_moran = (
    df_moran
    .fillna(
        {
            "qt_crimes": 0,
            "qt_roubos": 0,
            "qt_furtos": 0
        }
    )
)

# ============================================================
# 5. Criar taxas para análise espacial
# ============================================================

df_moran = (
    df_moran
    .withColumn(
        "tx_crimes_1000_hab",
        F.when(
            F.col("qt_pessoas") > 0,
            (F.col("qt_crimes") / F.col("qt_pessoas")) * 1000
        )
    )
    .withColumn(
        "tx_roubos_1000_hab",
        F.when(
            F.col("qt_pessoas") > 0,
            (F.col("qt_roubos") / F.col("qt_pessoas")) * 1000
        )
    )
    .withColumn(
        "tx_furtos_1000_hab",
        F.when(
            F.col("qt_pessoas") > 0,
            (F.col("qt_furtos") / F.col("qt_pessoas")) * 1000
        )
    )
    .withColumn(
        "log_tx_crimes_1000_hab",
        F.log1p(F.col("tx_crimes_1000_hab"))
    )
)

# ============================================================
# 6. Selecionar apenas colunas necessárias para QGIS / Moran
# ============================================================

df_moran_export = (
    df_moran
    .select(
        "cd_setor",
        "nm_dist",
        "nm_mun",
        "qt_pessoas",
        "area_km2",
        "qt_crimes",
        "qt_roubos",
        "qt_furtos",
        "tx_crimes_1000_hab",
        "tx_roubos_1000_hab",
        "tx_furtos_1000_hab",
        "log_tx_crimes_1000_hab",
        "indice_infraestrutura_urbana",
        "indice_mobilidade_urbana",
        "indice_caminhabilidade",
        "indice_vulnerabilidade_urbana",
        "log_renda_media",
        "densidade_populacional"
    )
)

# ============================================================
# 7. Validações
# ============================================================

print("Setores no Censo com população >= 100:")
print(
    df_censo
    .filter(F.col("nm_mun") == "São Paulo")
    .filter(F.col("qt_pessoas") >= 100)
    .select("cd_setor")
    .distinct()
    .count()
)

print("Setores no df_moran_export:")
print(
    df_moran_export
    .select("cd_setor")
    .distinct()
    .count()
)

df_moran_export.select(
    "tx_crimes_1000_hab"
).summary(
    "count",
    "mean",
    "stddev",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show(truncate=False)

df_moran_export.orderBy(
    F.col("tx_crimes_1000_hab").desc()
).show(20, truncate=False)

Setores no Censo com população >= 100:
25828
Setores no df_moran_export:
25828
+-------+------------------+
|summary|tx_crimes_1000_hab|
+-------+------------------+
|count  |25828             |
|mean   |26.847194033363458|
|stddev |60.57294332389427 |
|min    |0.0               |
|25%    |3.1545741324921135|
|50%    |11.363636363636363|
|75%    |28.225806451612904|
|max    |2525.862068965517 |
+-------+------------------+

+---------------+---------------+---------+----------+---------+---------+---------+---------+------------------+------------------+------------------+----------------------+----------------------------+------------------------+----------------------+-----------------------------+------------------+----------------------+
|cd_setor       |nm_dist        |nm_mun   |qt_pessoas|area_km2 |qt_crimes|qt_roubos|qt_furtos|tx_crimes_1000_hab|tx_roubos_1000_hab|tx_furtos_1000_hab|log_tx_crimes_1000_hab|indice_infraestrutura_urbana|indice_mobilidade_urbana|indice_caminhabilida

In [14]:
df_moran.select(
    "tx_crimes_1000_hab"
).summary().show()

+-------+------------------+
|summary|tx_crimes_1000_hab|
+-------+------------------+
|  count|             21029|
|   mean| 32.97395632192286|
| stddev| 65.60795883241144|
|    min|0.5931198102016608|
|    25%| 7.194244604316547|
|    50%|15.873015873015872|
|    75%| 34.48275862068965|
|    max| 2525.862068965517|
+-------+------------------+



In [21]:
pdf_moran_export = df_moran_export.toPandas()

pdf_moran_export.to_csv(
    "../output/moran_setores.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)